# Experiment: agentic retrieval with Qwen3.5-4B tool calling

Instead of searching with the user's raw request, [Qwen3.5-4B](https://huggingface.co/Qwen/Qwen3.5-4B) gets a
`search_recipes` tool and plans the searches: it translates non-English requests, extracts the dish or key
ingredients from vague ones and adds alternative phrasings. Each tool call runs the improved hybrid search; the
rankings are fused (RRF) with the original query's ranking, reranked with the Nemotron reranker, and the agent answers
in the user's language from the tool results. `think=True` enables Qwen3.5's reasoning before the tool calls.

Code: [`mmrag/agent.py`](../mmrag/agent.py). Results: [REPORT.md](../REPORT.md).

## Install dependencies

In [ ]:
!pip install -q -U "transformers>=5.2" "huggingface_hub>=1.0" "accelerate>=1.12" "datasets>=4.4" bm25s flash-linear-attention

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/ehzawad/multimodal-rag-nemotron"
for candidate in [Path.cwd(), Path.cwd().parent, Path("multimodal-rag-nemotron"), Path("/content/mmrag")]:
    if (candidate / "mmrag" / "agent.py").exists():
        REPO_DIR = candidate.resolve()
        break
else:
    !git clone -q -b exp-agentic {REPO_URL}
    REPO_DIR = Path("multimodal-rag-nemotron").resolve()
sys.path.insert(0, str(REPO_DIR))
print(f"[INFO] Using pipeline code from: {REPO_DIR}")

## Load models

In [ ]:
%%time
import torch
from datasets import load_dataset

from mmrag import agent as A, pipeline as P

dataset = load_dataset(P.DATASET_PATH, split="train")
rag = P.MultimodalRAG(dataset, P.load_embed_model(), P.load_document_embeddings(), *P.load_rerank_model(),
                      *P.load_generation_model())
agent = A.AgenticRAG(rag)
thinking_agent = A.AgenticRAG(rag, think=True)
print(f"[INFO] GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## What the agent searches for

In [ ]:
requests = [
    "banana bread with sour cream",
    "pan de plátano con crema agria",
    "টক দই দিয়ে কলার রুটি",
    "酸奶油香蕉面包",
    "something warm and filling for a rainy evening, no meat",
]
for request in requests:
    result = agent.search(query_text=request, k=20, rerank_top_n=20)
    print(f"{request}\n    searches: {result.timings['searches']}\n    top 3:    {[dataset[i]['name'] for i in result.indices[:3]]}"
          f"\n    timings:  plan {result.timings['plan_s']}s, rerank {result.timings.get('rerank_s')}s")

## Answer in the user's language

In [ ]:
%%time
request = "টক দই দিয়ে কলার রুটি"
result = agent.search(query_text=request, k=20, rerank_top_n=20)
print(agent.answer(request, result))

## With reasoning (thinking mode)

In [ ]:
%%time
request = "something warm and filling for a rainy evening, no meat"
searches, raw = thinking_agent.plan_searches(request)
print(raw[:1500])
print("searches:", searches)

## Evaluation

In [ ]:
results_file = REPO_DIR / "results" / "eval_results_agentic.md"
if results_file.exists():
    from IPython.display import Markdown, display
    display(Markdown(results_file.read_text()))
else:
    print("[INFO] Run: python scripts/eval_agentic.py")